In [19]:
import os
import h5py
import numpy as np
from PIL import Image
from collections import defaultdict
from tqdm import tqdm

def GetRGBDepthPathPairs(base):
    u = defaultdict(dict)
    # For each file in the directory
    for root, dirs, files in os.walk(base):
        for file in files:
            filepath = os.path.join(root, file)
            res = filepath.split('/')
            # Create a dict key to uniquely identify frames
            r1 = res[3] + '_' + res[4] + '_' + res[-1].split('.')[1]

            # Assumption is for every rgb there is a depth
            if file.lower().endswith(('.hdf5', '.hdf')):
                u[r1]['depth'] = filepath
            elif file.lower().endswith(('.jpg', '.jpeg')):
                u[r1]['rgb'] = filepath

    # Convert the default dict to a list
    pairs = []
    # Here implicitly it will be checked if every pair has a valid depth and rgb path, else there should be an error
    for k, v in u.items():
        pairs.append({
            'rgb': v['rgb'],
            'depth': v['depth']
        })

    return pairs


base = './Hypersim/'
train = base + 'Train/'
test = base + 'Test/'

train_pairs = GetRGBDepthPathPairs(train)
test_pairs = GetRGBDepthPathPairs(test)

In [20]:
print(len(train_pairs), len(test_pairs))

2000 100


In [27]:
H = []
W = []

global_min = float("inf")
global_max = float("-inf")

pixels_gt_10m = 0

for i in tqdm(train_pairs):
    depth_file = i['depth']
    rgb_file = i['rgb']
    with h5py.File(depth_file, 'r') as f:
        # Replace with the correct dataset key if needed
        depth = f[list(f.keys())[0]][:]

    # Load RGB
    rgb = np.array(Image.open(rgb_file))

    H.append(depth.shape[0])
    W.append(depth.shape[1])

    global_min = min(global_min, np.array(depth).min())
    global_max = max(global_max, np.array(depth).max())

    pixels_gt_10m += (depth > 30).sum()

print(set(H), set(W))
print('Min-Max: ', global_min, global_max, pixels_gt_10m)

100%|██████████| 2000/2000 [00:22<00:00, 87.10it/s]

{768} {1024}
Min-Max:  0.04437 45.25 201543


In [22]:
H = []
W = []

global_min = float("inf")
global_max = float("-inf")

for i in tqdm(test_pairs):
    depth_file = i['depth']
    rgb_file = i['rgb']
    with h5py.File(depth_file, 'r') as f:
        # Replace with the correct dataset key if needed
        depth = f[list(f.keys())[0]][:]

    # Load RGB
    rgb = np.array(Image.open(rgb_file))

    H.append(depth.shape[0])
    W.append(depth.shape[1])

    global_min = min(global_min, np.array(depth).min())
    global_max = max(global_max, np.array(depth).max())

print(set(H), set(W))
print('Min-Max: ', global_min, global_max)

100%|██████████| 100/100 [00:01<00:00, 96.05it/s]

{768} {1024}
Min-Max:  0.2489 27.33


In [23]:
depth_file = train_pairs[0]['depth']
rgb_file = train_pairs[0]['rgb']

# Load depth
with h5py.File(depth_file, 'r') as f:
    print("Keys:", list(f.keys()))

    # Replace with the correct dataset key if needed
    depth = f[list(f.keys())[0]][:]

# Load RGB
rgb = np.array(Image.open(rgb_file))

Keys: ['dataset']


In [28]:
print(rgb.shape, depth.shape)

(768, 1024, 3) (768, 1024)


In [24]:
img_uint8 = (rgb).clip(0, 255).astype(np.uint8)

img = Image.fromarray(img_uint8)
img.save('./resources/hypersim_rgb.png')

In [25]:
img = depth
img = img * 10000.0 
img = img.astype(np.uint16)
img = Image.fromarray(img)
img.save('./resources/hypersim_depth.png')

/tmp/ipykernel_64483/1985932730.py:2: RuntimeWarning: overflow encountered in multiply
  img = img * 10000.0
/tmp/ipykernel_64483/1985932730.py:3: RuntimeWarning: invalid value encountered in cast
  img = img.astype(np.uint16)
